# 04.7 Constants and Naming Conventions

Python has **no constants**. There is no `const` keyword and no way to stop a
name being rebound. What exists instead is a convention so widely followed that
it functions almost as well — plus a few genuine enforcement mechanisms when you
need them.

## Theory

### The convention

An `UPPER_SNAKE_CASE` name means: *this value is fixed; do not rebind it.*

```python
MAX_RETRIES = 3
DEFAULT_TIMEOUT = 30.0
API_BASE_URL = "https://api.example.com"
```

Python will happily let you reassign these. Nothing stops you. The name is a
message to other programmers, and linters will flag violations.

### Why no real constants?

It fits the language's philosophy — "we are all consenting adults here". Python
consistently prefers convention and clear communication over enforcement. The
same reasoning explains `_private` (04.5, Chapter 25).

### What you *can* actually enforce

<table>
<tr><th>Approach</th><th>Enforces?</th><th>Use when</th></tr>
<tr><td><code>UPPER_CASE</code> name</td><td>No — convention only</td><td>Almost always</td></tr>
<tr><td><code>Final</code> type hint</td><td>Type checker only</td><td>You run mypy</td></tr>
<tr><td><code>enum.Enum</code></td><td>Yes, at runtime</td><td>A fixed set of related values</td></tr>
<tr><td>Module-level immutable</td><td>The value, not the name</td><td>Tuples, frozensets</td></tr>
<tr><td><code>@property</code> with no setter</td><td>Yes, for attributes</td><td>Class attributes</td></tr>
</table>

### Magic numbers

The real purpose of a constant is not immutability — it is **naming**. A bare
`0.18` in the middle of a calculation tells the reader nothing. `TAX_RATE` tells
them everything, and means you change it in one place.

In [ ]:
# Constants are a convention. Python does not enforce them.
MAX_RETRIES = 3
TAX_RATE = 0.18
API_BASE_URL = "https://api.example.com"
SUPPORTED_FORMATS = ("json", "csv", "xml")

print("Declared constants:")
print("   MAX_RETRIES      =", MAX_RETRIES)
print("   TAX_RATE         =", TAX_RATE)
print("   API_BASE_URL     =", API_BASE_URL)
print("   SUPPORTED_FORMATS=", SUPPORTED_FORMATS)

# Nothing prevents rebinding.
MAX_RETRIES = 999
print("")
print("After MAX_RETRIES = 999:", MAX_RETRIES)
print("Python allowed it. Only a linter would object.")

# Restore it.
MAX_RETRIES = 3

## Why constants matter: magic numbers

The value of a constant is mostly in the **name**.

In [ ]:
# WITHOUT constants - what do these numbers mean?
def calculate_bad(amount, weight):
    """Work out a total. Unreadable and unmaintainable."""
    if weight > 30:
        return amount * 1.18 + 250
    return amount * 1.18 + 50


# WITH constants - the same logic, self-documenting.
TAX_RATE = 0.18
HEAVY_ITEM_THRESHOLD_KG = 30
HEAVY_SHIPPING_COST = 250
STANDARD_SHIPPING_COST = 50


def calculate_good(amount, weight_kg):
    """Work out a total including tax and shipping."""
    # The tax applies regardless of weight.
    with_tax = amount * (1 + TAX_RATE)

    # Heavier parcels cost more to ship.
    if weight_kg > HEAVY_ITEM_THRESHOLD_KG:
        return with_tax + HEAVY_SHIPPING_COST

    return with_tax + STANDARD_SHIPPING_COST


print("Both give the same answer:")
print("   bad( 100, 40):", round(calculate_bad(100, 40), 2))
print("   good(100, 40):", round(calculate_good(100, 40), 2))

print("")
print("But only one tells you WHY 30 and 250 matter - and only one")
print("lets you change the tax rate in a single place.")

## Enforcement option 1: `Final`

`typing.Final` tells a type checker the name must not be rebound. Python itself
ignores it entirely.

In [ ]:
from typing import Final

# Final is a promise to the type checker, not to the interpreter.
MAX_CONNECTIONS: Final = 100
SERVICE_NAME: Final[str] = "auth-service"

print("MAX_CONNECTIONS =", MAX_CONNECTIONS)
print("SERVICE_NAME    =", SERVICE_NAME)

# Python still allows rebinding at runtime.
MAX_CONNECTIONS = 200
print("")
print("After rebinding at runtime:", MAX_CONNECTIONS)
print("Python did not complain.")

print("")
print("But mypy WOULD report:")
print('   error: Cannot assign to final name "MAX_CONNECTIONS"')
print("")
print("So Final is useful only if you actually run a type checker.")
print("Chapter 32 covers type checking.")

## Enforcement option 2: `Enum`

For a fixed set of related values, an enum gives real runtime protection and
better error messages.

In [ ]:
from enum import Enum

class Status(Enum):
    """The states an order can be in."""
    PENDING = "pending"
    SHIPPED = "shipped"
    DELIVERED = "delivered"


# Members are accessed by name.
print("Status.PENDING      :", Status.PENDING)
print("   .name            :", Status.PENDING.name)
print("   .value           :", Status.PENDING.value)

# Enum members CANNOT be reassigned.
try:
    Status.PENDING = "something else"
except AttributeError as error:
    print("")
    print("Reassigning a member:", error)

# Iterating gives every valid value - useful for validation.
print("")
print("All valid statuses:")
for status in Status:
    print("   ", status.name.ljust(10), status.value)

# A typo produces an immediate, clear error.
print("")
try:
    Status("shippd")
except ValueError as error:
    print("Invalid value:", error)

print("")
print("Compare with a plain string, where a typo fails silently.")
print("Enums are covered fully in Chapter 28.")

## Enforcement option 3: immutable values

You cannot stop the name being rebound, but you can stop the *value* being
modified.

In [ ]:
# A mutable constant is a contradiction - anyone can change it.
MUTABLE_DEFAULTS = {"retries": 3, "timeout": 30}
MUTABLE_DEFAULTS["retries"] = 999

print("A dict 'constant' after tampering:", MUTABLE_DEFAULTS)

# An immutable value resists modification.
IMMUTABLE_FORMATS = ("json", "csv", "xml")

try:
    IMMUTABLE_FORMATS[0] = "yaml"
except TypeError as error:
    print("")
    print("Modifying a tuple constant:", error)

# frozenset for an unordered collection.
ALLOWED_METHODS = frozenset({"GET", "POST", "PUT"})

try:
    ALLOWED_METHODS.add("DELETE")
except AttributeError as error:
    print("Modifying a frozenset constant:", error)

print("")
print("GUIDANCE: prefer tuple over list, and frozenset over set,")
print("for anything declared as a constant.")

# MappingProxyType gives a read-only view of a dict.
from types import MappingProxyType

_config = {"retries": 3, "timeout": 30}
CONFIG = MappingProxyType(_config)

print("")
print("Read-only dict view:", dict(CONFIG))
try:
    CONFIG["retries"] = 999
except TypeError as error:
    print("Modifying it:", error)

## Naming conventions recap

These came up in 03.5. Here they are applied to real values.

In [ ]:
conventions = [
    ("MAX_RETRIES", "UPPER_SNAKE", "module constant"),
    ("user_count", "snake_case", "variable"),
    ("calculate_total", "snake_case", "function"),
    ("BankAccount", "PascalCase", "class"),
    ("ValidationError", "PascalCase", "exception"),
    ("_internal_cache", "_leading", "internal, not part of the API"),
    ("__mangled", "__double", "name-mangled inside a class"),
    ("class_", "trailing_", "avoiding a keyword clash"),
    ("_", "underscore", "a value you will not use"),
]

print("Name                Convention      Means")
print("-" * 60)
for name, convention, means in conventions:
    print(name.ljust(20), convention.ljust(15), means)

print("")
print("For constants specifically:")
rules = [
    "Define them at module level, near the top",
    "Use a name saying what it limits or configures",
    "Prefer immutable types - tuple over list, frozenset over set",
    "Group related values into an Enum",
    "Never compute them from mutable state",
]
for index, rule in enumerate(rules, start=1):
    print("  ", index, "-", rule)

## Takeaways

1. Python has **no real constants** — `UPPER_SNAKE_CASE` is a convention
   communicating intent, not a guarantee.
2. The main value of a constant is the **name**, which removes magic numbers and
   centralises the value.
3. `typing.Final` is enforced by **type checkers only**, not at runtime.
4. `Enum` gives genuine runtime protection and rejects invalid values clearly.
5. Use **immutable types** for constants: `tuple` not `list`, `frozenset` not
   `set`, `MappingProxyType` for a read-only dict.
6. Define constants at module level, near the top of the file.

## Try it yourself

1. Find a magic number in your own code and replace it with a named constant.
2. Declare a `Final` constant and rebind it. Does Python object? Would mypy?
3. Convert a set of related string values into an `Enum`. Try an invalid value.
4. Wrap a config dict in `MappingProxyType` and attempt to modify it.